In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)

In [ ]:
from PIL import Image
from typing import Callable
from intecomm_analytics.dataframes import get_df_main_1858
from intecomm_analytics.constants import UNDEFINED, DM_ALONE, HTN_ALONE, HIV_ALONE, HTN_DM
from intecomm_rando.constants import COMMUNITY_ARM, FACILITY_ARM
from intecomm_analytics.utils import (
    get_great_table,
    get_vl, get_glucose,
    get_bp,
    get_bp_high,
    get_columns_for_days_to_event,
    get_primary_cohorts_by_categorical_column,
)

In [ ]:
df_main_orig = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_main_orig.query("offstudy_reason=='pregnant'")["assignment"]

In [ ]:
df_main = df_main_orig[df_main_orig.retained_12m==1].copy()
df_main = df_main.reset_index(drop=True)

In [ ]:
assert len(df_main.query("primary_cohort.isin([1,2,3,4])")) == 1600

In [ ]:
footnotes = []

def get_footnote(df:pd.DataFrame, col:str, condition:Callable, label:str, sup:str)->tuple[str,str] | None:
    df = df[condition(df) & df[col].isna()]
    if len(df)>0:
        return f"{label}<sup>{sup}</sup>", f"{sup} missing {len(df)}"
    return label, ""

In [ ]:
# primary_cohort
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "primary_cohort")
dftbl = pd.DataFrame(tbl_dct)
mapping = {DM_ALONE:"Diabetes alone", HTN_ALONE:"Hypertension alone", HTN_DM:"Diabetes and hypertension", HIV_ALONE:"HIV alone", UNDEFINED:"UNDEFINED", "n":"n"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl = dftbl[dftbl["Statistics"]!="UNDEFINED"]
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Diabetes alone", "Hypertension alone", "Diabetes and hypertension", "HIV alone"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (0.0%)", "NA")
dfnum = dftbl.iloc[0:1]
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
df_primary_cohort = dftbl.copy()

In [ ]:
# country
tbl_dct = get_primary_cohorts_by_categorical_column(df_main, "country")
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", "TZ": "Tanzania", "UG": "Uganda"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Tanzania", "Uganda"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "country"
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfcountry = dftbl.copy()

In [ ]:
# bp_controlled_endline
col = "bp_controlled_endline"
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension"
cohort_cond = lambda _df: ((_df.primary_cohort==HTN_ALONE) | (_df.primary_cohort==HTN_DM))
func = get_bp

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "**")
if footnote:
    footnotes.append(footnote)
df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond(df1), label)

df_htn_bp_end = dftbl.copy()

In [ ]:
# bp_controlled_endline (alone)
col = "bp_controlled_endline"
label = "<140/90 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension alone"
cohort_cond = lambda _df: (_df.primary_cohort==HTN_ALONE)
func = get_bp

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "*")
if footnote:
    footnotes.append(footnote)
df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

df_htn_alone_bp_end = dftbl.copy()

In [ ]:
col = "bp_severe_htn_endline"
label = ">180/120 mm Hg among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;hypertension"
cohort_cond = lambda _df: (_df.primary_cohort.isin([HTN_ALONE, HTN_DM]))
func = get_bp_high

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "*&Dagger;")
if footnote:
    footnotes.append(footnote)
df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

df_htn_bp_high_end = dftbl[~dftbl.Statistics.isna()].copy()

In [ ]:
# glucose_controlled_endline alone
col = "glucose_controlled_endline"
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes alone"
cohort_cond = lambda _df: (_df.primary_cohort==DM_ALONE)
func = get_glucose

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "&Dagger;")
if footnote:
    footnotes.append(footnote)
df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

df_dm_alone_glu_end = dftbl.copy()

In [ ]:
# glucose_controlled_endline
col = "glucose_controlled_endline"
label = "<7.0 mmol/L among<BR>&nbsp;&nbsp;&nbsp;&nbsp;participants with<BR>&nbsp;&nbsp;&nbsp;&nbsp;diabetes"
cohort_cond = lambda _df: ((_df.primary_cohort==DM_ALONE) | (_df.primary_cohort==HTN_DM))
func = get_glucose

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "&Dagger;&Dagger;")
if footnote:
    footnotes.append(footnote)
df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

df_dm_glu_end = dftbl.copy()

In [ ]:
# bp_glucose_composite
col = "primary_composite_endline"
label = "BP/Glucose controlled composite"
cohort_cond = lambda _df: ((_df.primary_cohort==DM_ALONE) | (_df.primary_cohort==HTN_ALONE) | (_df.primary_cohort==HTN_DM))

df1 = df_main.copy()
label, footnote = get_footnote(df1, col, cohort_cond, label, "&sect;")
if footnote:
    footnotes.append(footnote)

df1 = df_main[df_main[col].notna()].copy()
df1.loc[cohort_cond(df1), col] = df1.loc[cohort_cond(df1), col].fillna(-1)

tbl_dct = get_primary_cohorts_by_categorical_column(df1[cohort_cond(df1)], col)
dftbl = pd.DataFrame(tbl_dct)
mapping = {"n": "n", -1: "Missing", 1: label, 0: "Uncontrolled"}
dftbl["Statistics"] = dftbl["Statistics"].map(mapping)
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", label, "Uncontrolled", "Missing"], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.reset_index(drop=True)
for col in ["Community Ncd", "Facility Ncd"]:
    value = dftbl.loc[1, col].split(" ")
    value = [value[0], "/", str(dftbl.loc[0, col]), " ", value[1]]
    value = "".join(value)
    dftbl.loc[1, col] = value
dftbl.replace("0 (0.0%)", "NA", inplace=True)
dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)

dfcomposite = dftbl.copy()

In [ ]:
# df1[[col for col in df1.columns if col.startswith("primary")]]

In [ ]:
# vl_controlled_endline
col = "vl_controlled_endline"
label = "<1000 copies per mL"
cohort_cond = lambda _df: ((_df.hiv==1) & (_df.dm==0) & (_df.htn==0))
func = get_vl

df1 = df_main.copy()
# label, footnote = get_footnote(df1, col, cohort_cond, label, "+")
# if footnote:
#     footnotes.append(footnote)
# df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

dfvl_end = dftbl.copy()

In [ ]:
# vl_controlled_endline_400
col = "vl_controlled_endline_400"
label = "<400 copies per mL"
cohort_cond = lambda _df: ((_df.hiv==1) & (_df.dm==0) & (_df.htn==0))
func = get_vl

df1 = df_main.copy()
# label, footnote = get_footnote(df1, col, cohort_cond, label, "++")
# if footnote:
#     footnotes.append(footnote)
# df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

dfvl400_end = dftbl.copy()

In [ ]:
# vl_controlled_endline_50
col = "vl_controlled_endline_50"
label = "<50 copies per mL"
cohort_cond = lambda _df: ((_df.hiv==1) & (_df.dm==0) & (_df.htn==0))
func = get_vl

df1 = df_main.copy()
# label, footnote = get_footnote(df1, col, cohort_cond, label, "+++")
# if footnote:
#     footnotes.append(footnote)
# df1 = df1[df1[col].notna()].copy()
dftbl= func(df1, col, cohort_cond, label)

dfvl50_end = dftbl.copy()

In [ ]:
dfvlall_end = pd.concat([dfvl_end, dfvl400_end, dfvl50_end])
dfvlall_end = dfvlall_end.reset_index(drop=True)

In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (df_primary_cohort, ["Condition"]),
    (dfvlall_end, ["HIV viral load endline"]),
    (df_htn_alone_bp_end, ["Blood pressure endline"]),
    (df_htn_bp_end, ["Blood pressure endline"]),
    (df_htn_bp_high_end, ["Blood pressure endline"]),
    (df_dm_alone_glu_end, ["Fasting blood glucose endline"]),
    (df_dm_glu_end, ["Fasting blood glucose endline"]),
    (dfcomposite, ["BP/Glucose controlled composite"])
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
footnotes.sort()
outcomes_table = get_great_table(
    dftbl_final,
    group_row_headers,
    f"Table 1.2: Endline clinical outcomes (retained 12m)",
    source_notes="<BR>".join(footnotes),
)
outcomes_table.show()

In [ ]:
# save as png
outcomes_table.save(analysis_folder / f"endline_clinical_measures_12m.png")
# export to PDF
image = Image.open(analysis_folder / f"endline_clinical_measures_12m.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / f"endline_clinical_measures_12m.pdf", "PDF", resolution=800, optimize=True, quality=95)

In [ ]:
df1 = df_main.copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["onstudy_bins"] = pd.cut(df1["onstudy_days"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "onstudy_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Mean, SD"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        data.append(
            f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['onstudy_days'].mean(),1)} "
            f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['onstudy_days'].std(),1)})"
        )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Mean, SD", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfonstudy= dftbl.copy()

In [ ]:
df1 = df_main[df_main.primary_cohort.isin([HIV_ALONE])].copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["vl_days_to_event_bins"] = pd.cut(df1["vl_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "vl_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['vl_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfvldays= dftbl.copy()

In [ ]:
df1 = df_main[df_main.primary_cohort.isin([HTN_ALONE, HTN_DM])].copy()
bins = [0, 181, 269, 365, 1000]
labels = ["<182", "182 to <270", "270 to <365", ">=365"]
df1["bp_days_to_event_bins"] = pd.cut(df1["bp_days_to_event"], bins, labels=labels)
tbl_dct = get_primary_cohorts_by_categorical_column(df1, "bp_days_to_event_bins")
dftbl = pd.DataFrame(tbl_dct)
data = ["Median, (min-max)"]
for col in ["ncd", "hiv_only"]:
    for arm in [COMMUNITY_ARM, FACILITY_ARM]:
        if df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].isna().all():
            data.append("0 (0-0)")
        else:
            data.append(
                f"{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].median(),1)} "
                f"({round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].min(),1)}-{round(df1[(df1.assignment==arm) & (getattr(df1, col)==1)]['bp_days_to_event'].max(),1)})"
            )
dftbl.loc[len(dftbl)] = data
dftbl["Statistics"] = pd.Categorical(dftbl["Statistics"], categories=["n", "Median, (min-max)", *labels], ordered=True)
dftbl = dftbl.sort_values(by=["Statistics"], ascending=True)
dftbl = dftbl.replace("0 (nan%)", "0 (0.0%)")
# dftbl["variable"] = "age_in_years"
# dftbl = dftbl.drop(0)
dftbl = dftbl.reset_index(drop=True)
dfbpdays= dftbl.copy()

In [ ]:
df1 = df_main[df_main.primary_cohort.isin([DM_ALONE, HTN_DM])].copy()
dftbl = get_columns_for_days_to_event(df1, "glucose_days_to_event", set_zero_to_na=["hiv"])
dfgludays= dftbl.copy()

In [ ]:
df1

In [ ]:
groupings = [
    (dfnum, [""]),
    (dfcountry, ["Site"]),
    (dfbpdays, ["Blood pressure"]),
    (dfgludays, ["Fasting blood glucose"]),
    (dfvldays, ["HIV viral load"]),
]
group_row_headers = [(df, row_headers * (len(df))) for df, row_headers in groupings]
group_row_headers = [row_heading for _, row_headers in group_row_headers for row_heading in row_headers]
# concat all
dftbl_final = pd.concat([df for df, _ in groupings])
dftbl_final = dftbl_final.reset_index(drop=True)
# convert to GT
outcomes_table = get_great_table(dftbl_final, group_row_headers, f"Table 1.3: Days to endline clinical outcomes (retained 12m)")
outcomes_table.show()

In [ ]:
# save as png
outcomes_table.save(analysis_folder / f"days_to_endline_outcomes_12m.png")
# export to PDF
image = Image.open(analysis_folder / f"days_to_endline_outcomes_12m.png")
image = image.resize((image.width * 6, image.height * 6), Image.LANCZOS)
image.save(analysis_folder / f"days_to_endline_outcomes_12m.pdf", "PDF", resolution=800, optimize=True, quality=95)